In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pyiron_base import Project

In [ ]:
from pyiron_glass import (
    melt_quench_simulation,
    generate_potential,
    get_ase_structure,
    get_structure_dict,
)

In [ ]:
pr = Project("test")
atoms_dict = get_structure_dict(
    comp="0.25CaO-0.25Al2O3-0.50SiO2", 
    n_molecules=200, 
    density=2.69*1.0, 
    min_distance=1.8, 
    max_attempts_per_atom = 10000,
    pyiron_project=pr,
)
structure = get_ase_structure(atoms_dict=atoms_dict, pyiron_project=pr,)
potential = generate_potential(atoms_dict=atoms_dict, pyiron_project=pr,)

In [ ]:
delayed = melt_quench_simulation(
    structure=structure, 
    potential=potential,
    temperature_high=5000,
    temperature_low=300,
    n_print=1000,
    working_directory="lmp_tmp_directory",
    heating_rate=int(1e14),
    cooling_rate=int(1e14),
    pyiron_project=pr,
)

In [ ]:
delayed.draw()

In [ ]:
result = delayed.pull() 

In [ ]:
mean_temp = np.mean(result["temperature"])
print(f"{mean_temp:.1f} K")


In [ ]:
plt.plot(result["steps"]*1e-3, result["temperature"])
plt.axhline(np.mean(result["temperature"]), color="red", linestyle="--")
plt.xlabel("Time [ps]")
plt.ylabel("Temperature [K]");

In [ ]:
print(result["structure"])

In [ ]:
from pyiron_glass import mass


V = np.mean(result["generic"]["volume"])*1e-24 # volume in cm#


massO = mass.get_atomic_mass("O")
massSi = mass.get_atomic_mass("Si")
massAl = mass.get_atomic_mass("Al")
massCa = mass.get_atomic_mass("Ca")

Mol = 200
massSiO2  = massSi + 2*massO # g/mol
massAl2O3 = 2*massAl + 3*massO
massCaO   = massCa + massO

massTot = ((0.25*200)*massCaO + (0.25*200)*massAl2O3 + (0.5*200)*massSiO2)/6.02214076e23


print(f"{(massTot/V):.3f} g/cm3") 


